In [9]:
import urllib.request as request
import http.cookiejar as cookiejar
import urllib.error
import urllib.parse as parse

In [10]:
_WEB_URL_INSIDER = "https://www.barchart.com/stocks/quotes"
API_URL = 'https://www.barchart.com/proxies/core-api/v1/quotes/get?fields=symbol%2Cexchange%2CsymbolName%2CsymbolType%2CbaseCode%2CsymbolCode%2CpreviousPrice%2CpreviousHighPrice%2CpreviousLowPrice%2CweeklyPreviousPrice%2CweeklyPreviousHighPrice%2CweeklyPreviousLowPrice%2CmonthlyPreviousPrice%2CmonthlyPreviousHighPrice%2CmonthlyPreviousLowPrice%2CchartTime.format(Y-m-d%5CTH%3Ai%3As)%2CshortSymbol%2ChasBats%2ChasJerq%2ClastPrice%2CpercentChange%2CpriceChange%2CopenPrice%2ClowPrice%2ChighPrice%2CpointValue%2ChighPrice1y%2ClowPrice1y&raw=1&meta=symbol.normalized&symbols=MSFT'


def user_agent(webkit_version: str = "537.36", chrome_version: str = "129.0.0.0", os: str = "Macintosh; Intel Mac OS X 10_15_7"):
    return f'Mozilla/5.0 ({os}) AppleWebKit/{webkit_version} (KHTML, like Gecko) Chrome/{chrome_version} Safari/{webkit_version}'

def _weburl(symbol: str, section: str = 'insider-trades'):
    return f"{_WEB_URL_INSIDER}/{symbol}/{section}"

In [11]:
req = request.Request(_weburl("MSFT"),headers={'User-Agent': user_agent()})
resp = request.urlopen(req)

In [12]:
print(resp.info())

Content-Type: text/html; charset=UTF-8
Transfer-Encoding: chunked
Connection: close
Date: Sat, 02 Nov 2024 05:47:23 GMT
Strict-Transport-Security: max-age=31536000;
Content-Encoding: gzip
Server: nginx
Vary: Accept-Encoding
Cache-Control: no-cache, no-store, max-age=0, must-revalidate
Expires: Fri, 01 Jan 1990 00:00:00 GMT
Set-Cookie: laravel_token=eyJpdiI6ImdYRHJBems4THRqZHlIVkxHamZ4b1E9PSIsInZhbHVlIjoiK0p2N2lEZWJJZWM4SE5qV0ZYb3FuOVFRaWhLOWxlT1o5cFl2S2xDazc3cDVtVTNKdTJYcnZ2WWNQU0c4eXFBNXJueTR3NnE0NDBMdFhCYzUrNDFuRjFmMVpjL2tVZ2FTQ1V3dmNjMTFGNlo5TzVFYko0U2NuMEdTbE9hUjVUYTA1dFQ3cnRMWTVSeGV3MHl4ajQ5ZUJYQnluV0Z3dXVGWG5OOFEzUFFEaytOZ1NERjdDMEFrV2d6alN2NXlENE1nWWVweWg3VSsyVXNPUXd2MlZLSlhlcWh1QkJ1cXlNMUFXRGg3Tmh1bGRZZEZoSU5qVkRqOVltMWczV0l6Z2dqNXB3NWYwYVNhdzR1K254TmZkQU4yV2IzNUkzbFFrYjNFUE5Dbkx3bEV2emZqdERHM3NTYk5HN1B4UDRDYXZTYkgiLCJtYWMiOiJiZmFhYzc1ZThiYTM2MDI5ZDFiNWMyYzg3M2I4NWE4MDQ3N2U1NzEzNzBjNjViZDc3ZWFjYzUxNjQ4MjhiZjg1IiwidGFnIjoiIn0%3D; expires=Sat, 02-Nov-2024 07:47:23 GMT; Max-Age=72

In [13]:
# l_cookies = print(resp.info().get_all("Set-Cookie"))

In [14]:
cj = cookiejar.CookieJar()
cj.extract_cookies(resp, req)
cookies = cj.make_cookies(resp, req)
tok = None
for c in cookies:
    if c.name == "XSRF-TOKEN":
        tok = parse.unquote(c.value)
        break

In [15]:
opener = request.build_opener(request.HTTPCookieProcessor(cj))
# opener = request.build_opener()
opener.addheaders = [
    ('User-Agent', user_agent()),
    ('X-XSRF-TOKEN', tok),
    ('Referrer', _weburl("MSFT")),
    ('Connection', 'keep-alive'),
    ('Credentials', 'include'),
    ('Accept', 'application/json'),
    ('Host', 'www.barchart.com'),
    ('Accept-Encoding', 'gzip, deflate, br'),
]
request.install_opener(opener)

In [16]:

try:
    resp2 = request.urlopen(API_URL)
except urllib.error.HTTPError as e:
    print(e)
    with open("dump.txt", "wt") as f:
        f.write(e.read().decode("utf-8"))
else:
    resp2.info()